# March Mania · Shared-system ranking matchups
**Milestone 12 — recover an established ranking control → engineer two paired comparisons → measure incremental value.**

Milestone 11 completed; neither its primary nor secondary comparison passed the declared expansion rule. This notebook changes the information representation, not the tournament classifier. The published ranking consensus is an **established control**, not a new-feature claim. Two matched-system features must improve beyond that control.

Men only; 2016–2019 exploratory main-draw validation. Seven existing base snapshots and four saved references are reused. At most **16 new classifier fits**, **zero rating fits**, seven publication panels and seven matchup tables. No new source download, cloud action, Git write, package installation or automatic submission. Read `RESEARCH_PLAN.md`.

In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink
KIT = Path.cwd().resolve()
if not (KIT / 'run_round12.py').is_file():
    KIT = Path.home() / 'march_ranking_matchups'
assert (KIT / 'run_round12.py').is_file(), 'Open this notebook inside march_ranking_matchups.'
sys.path.insert(0, str(KIT))
from run_round12 import run_stage
from ranking_plots import figures
pio.renderers.default = 'plotly_mimetype'
print('Kernel:', sys.executable)
print('Research kit:', KIT)
print('No training occurs until the explicit evaluation cell.')

## 1. Preserve the completed result
The Huber primary improved one of four seasons. The secondary compression addition improved three, but its mean gain did not reach the declared threshold. Neither should be relabeled as a successful experiment because a different comparison looks best.

In [ ]:
display(pd.read_csv(KIT / 'evidence/round11/metrics.csv')[['Season','recipe','brier','delta_vs_anchor']].round(7))
print(json.dumps(json.loads((KIT / 'evidence/round11/decisions.json').read_text()), indent=2))
print('Those are supplied results, not new measurements.')

## 2. Build legal ranking panels and paired comparisons
Each system contributes its **latest whole edition** between day 118 and day 132. Missing teams are not backfilled from older editions. Normalization uses the system's own published rank cohort before restricting to NCAA seeds. All possible seeded pairings are built before attaching tournament outcomes.

**Control:** difference of the teams' marginal consensus logits, matching the existing repository definition.

**Candidate A:** median of within-system logit differences for systems covering **both** teams, minus the control difference. Median differences and differences of medians need not agree.

**Candidate B:** a smoothed log ratio of the shared systems ranking A above B versus B above A. True ordinal ties count half per side; clipped logits must not create false ties. One pseudo-vote is added to each side.

These are correlated opinions, not independent trials or calibrated win probabilities. Fixed minimum support is three shared systems. The ordinal file is streamed in bounded chunks; accepted per-season outputs are checkpointed. Preparation ceiling: **300 seconds**; heartbeats: **15 seconds**.

In [ ]:
run_stage('prepare', max_seconds=300)
RUN = Path(json.loads((KIT / 'reports/latest_run.json').read_text())['run_dir'])
print(json.dumps(json.loads((RUN / 'prepare.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'panel_coverage.csv'))
display(pd.read_csv(RUN / 'coverage.csv'))
display(pd.read_csv(RUN / 'prior_replay.csv').round(7))

In [ ]:
registry = pd.read_csv(RUN / 'feature_registry.csv')
display(registry.loc[registry.family.ne('reference'), ['feature','family','new_candidate','description']])
profiles = pd.read_csv(RUN / 'team_profiles.csv')
display(profiles.query('Season == 2019').sort_values('consensus_percentile', ascending=False).head(12))
print('Consensus is not new to the project. These displays do not prove feature value.')

## 3. Fixed comparisons, without replacing the classifier
| Configuration | Inputs | Role |
|---|---:|---|
| anchor | 16 | Replay four existing references |
| anchor_consensus | 17 | Established-information control |
| anchor_consensus_median | 18 | Shared-system median residual |
| anchor_consensus_votes | 18 | Shared-system agreement |
| anchor_consensus_both | 19 | Primary complete-family comparison |

Train on 2013 through the year before each validation year. Evaluate 2016, 2017, 2018 and 2019, 63 main-draw games per real-data year. The original logistic C=0.1, no intercept, mirrored orientations, physical-game weighting, and training-only RMS scaling remain unchanged. Only training-constant inputs may be removed. **16 new classifiers; evaluation ceiling 180 seconds.**

Primary: `pairwise_given_consensus`. The established control effect (`consensus_given_anchor`) is separately reported; it cannot substitute for failed new features. Drop-one comparisons hold all other columns fixed.

All four seasons are repeatedly-used development history. Any gain remains exploratory and needs a later-era/stronger-recipe test.

In [ ]:
run_stage('evaluate', max_seconds=180)
metrics = pd.read_csv(RUN / 'metrics.csv')
display(metrics[['Season','recipe','brier','log_loss','delta_vs_anchor','source']].round(7))
display(pd.read_csv(RUN / 'ablations.csv').round(7))
print(json.dumps(json.loads((RUN / 'decisions.json').read_text()), indent=2))
print(json.dumps(json.loads((RUN / 'evaluation_receipt.json').read_text()), indent=2))

## 4. Save the scientific report before inline rendering
The gate requires mean Brier change ≤ −0.0005, at least three of four seasons improving, and worst deterioration ≤ +0.003. These are resource-allocation rules, not significance tests. No automatic promotion or follow-on experiment.

A failed scientific gate is not a technical error: export the result. Reporting ceiling: **120 seconds**. The archive excludes raw rankings, individual predictions, models, pair-level rows and edited repository notebooks. All private checkpoints remain in place.

In [ ]:
run_stage('report', max_seconds=120)
record = json.loads((KIT / 'reports/latest_report.json').read_text())
print('Scientific report saved:', record['return_zip'])
display(FileLink(str(Path(record['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(record['html']).relative_to(KIT))))

## 5. Ten interactive charts
Review source coverage, shared-system support, paired discrepancies, fixed-model scores, add/drop effects, calibration, and training-only redundancy. More ranking systems does **not** imply proportionally more independent evidence.

In [ ]:
plots = figures(RUN, KIT / 'evidence/round11')
assert len(plots) == 10
for fig in plots:
    fig.show()

## 6. Return one report and stop
Save this notebook with **Ctrl+S**. Download `reports/milestone_12_return.zip`. It includes the prior milestone 11 archive for continuity. Keep every `private_runs` directory. Do not generate a submission or launch another feature test yet.

In [ ]:
print('Return:', record['return_zip'])
print('HTML:', record['html'])
print('Preserved private checkpoints:', RUN)
display(FileLink(str(Path(record['return_zip']).relative_to(KIT))))
print('No leaderboard score was measured by this notebook. Feature engineering remains open.')